# 07 — Construire un wheel avec `uv`

## Objectifs pédagogiques

À la fin de ce notebook, vous saurez :

- comprendre ce qu'est une *wheel* (`.whl`) ;
- construire votre package avec `uv build` ;
- installer un wheel localement ;
- savoir comment on publierait sur PyPI.

## Prérequis

- `pyproject.toml` (notebook 03) ;
- structure de projet avec `src/` layout ;
- pytest, ruff.

## Plan

1. Que veut dire « construire un package » ?
2. Wheel vs sdist
3. `uv build`
4. Contenu d'un wheel
5. Installer localement
6. Versionner
7. Publier sur PyPI (aperçu)
8. Synthèse
9. Exercices

---


## 1. Que veut dire « construire un package » ?

Le code dans `src/mon_projet/` doit être **empaqueté** dans un format installable pour qu'un autre développeur (ou une machine de prod) puisse l'installer avec `pip install mon-projet`.

Cette étape transforme :

```
src/mon_projet/*.py  +  pyproject.toml
```

en un fichier `.whl` (et/ou `.tar.gz`) prêt à être installé.

---


## 2. Wheel vs sdist

| Format | Extension | Contenu |
|---|---|---|
| **Wheel** | `.whl` | Package **pré-compilé**, installable directement |
| **sdist** | `.tar.gz` | **Sources** — à recompiler à l'installation |

On construit **les deux** pour distribuer sur PyPI : wheel pour la vitesse, sdist pour la reproductibilité.

---


## 3. `uv build`

```bash
uv build
```

Cette commande :

1. lit `pyproject.toml` ;
2. invoque le `build-backend` configuré (Hatchling par exemple) ;
3. produit un wheel et un sdist dans `dist/` :

```
dist/
├── mon_projet-0.1.0-py3-none-any.whl
└── mon_projet-0.1.0.tar.gz
```

### Alternative : `python -m build`

Sans `uv`, on peut installer `build` puis lancer `python -m build`. Même résultat.

---


## 4. Contenu d'un wheel

Un `.whl` est en réalité un zip. Il contient :

- le code Python (`.py`) ;
- des métadonnées (`METADATA`, version, dépendances) ;
- éventuellement des extensions C compilées ;
- des entry points (commandes CLI).

Le nom suit le format `{name}-{version}-{python_tag}-{abi}-{platform}.whl`. `py3-none-any` = Python 3 pur, sans extension binaire.

---


## 5. Installer localement

Pour tester avant de publier :

```bash
uv pip install dist/mon_projet-0.1.0-py3-none-any.whl
# ou
pip install dist/mon_projet-0.1.0-py3-none-any.whl
```

### Installation éditable (développement)

```bash
uv pip install -e .
# ou
pip install -e .
```

En mode éditable, les modifications du code source sont prises en compte **sans réinstaller**.

---


## 6. Versionner

La version vit dans `pyproject.toml` sous `[project] version = "0.1.0"`. Convention **SemVer** :

| Partie | Quand l'incrémenter |
|---|---|
| major (`X.y.z`) | Changement cassant |
| minor (`x.Y.z`) | Nouvelle fonctionnalité rétro-compatible |
| patch (`x.y.Z`) | Correction de bug |

Tant qu'on est en `0.x`, tout peut casser. À partir de `1.0`, on s'engage.

---


## 7. Publier sur PyPI (aperçu)

```bash
# 1. Builder
uv build

# 2. (La 1re fois) : créer un compte sur https://pypi.org/ et un token API

# 3. Publier
uv publish
# ou avec twine :
# twine upload dist/*
```

Tester d'abord sur **TestPyPI** : https://test.pypi.org/ — dépôt de staging, même API que PyPI.

---


## 8. Synthèse

| Commande | Effet |
|---|---|
| `uv build` | Construit wheel + sdist dans `dist/` |
| `uv pip install dist/*.whl` | Installe un wheel local |
| `uv pip install -e .` | Installation éditable |
| `uv publish` | Publie sur PyPI |

### Règles

1. Toujours builder **après** avoir tagué la version dans `pyproject.toml`.
2. Tester d'abord sur TestPyPI.
3. Commencer en `0.1.0` ; passer à `1.0.0` quand l'API est stable.
4. CI : builder sur chaque tag Git, publier automatiquement.

---


## 9. Exercices

### Exercice 1 — Contenu de `dist/` *(facile)*

Après un `uv build` sur un projet `mon-outil` en version `0.1.0`, quels fichiers attendez-vous dans `dist/` ? Écrire la liste.

In [ ]:
# Votre code ici


In [ ]:
# ▶ Une fois votre solution écrite ci-dessus, exécutez cette cellule
# pour signaler à votre formateur que vous avez tenté l'exercice.
import sys
from pathlib import Path
for _p in (Path.cwd(), *Path.cwd().parents):
    if (_p / "_common" / "utils_pedagogie.py").exists():
        sys.path.insert(0, str(_p / "_common")); break
from utils_pedagogie import marquer_tentative
marquer_tentative(notebook="07_Build_wheel", exercice=1)


<details>
<summary>📖 Voir la correction</summary>

```python
attendus = [
    'mon_outil-0.1.0-py3-none-any.whl',
    'mon_outil-0.1.0.tar.gz',
]
for f in attendus:
    print(f)
```

</details>

### Exercice 2 — Parse d'un nom de wheel *(facile)*

Écrire `info_wheel(nom: str) -> dict[str, str]` qui décompose un nom de wheel au format `name-version-python-abi-platform.whl` en un dict.

In [ ]:
# Votre code ici


In [ ]:
# ▶ Une fois votre solution écrite ci-dessus, exécutez cette cellule
# pour signaler à votre formateur que vous avez tenté l'exercice.
import sys
from pathlib import Path
for _p in (Path.cwd(), *Path.cwd().parents):
    if (_p / "_common" / "utils_pedagogie.py").exists():
        sys.path.insert(0, str(_p / "_common")); break
from utils_pedagogie import marquer_tentative
marquer_tentative(notebook="07_Build_wheel", exercice=2)


<details>
<summary>📖 Voir la correction</summary>

```python
def info_wheel(nom: str) -> dict[str, str]:
    """Extrait les composants d'un nom de wheel."""
    base = nom.removesuffix('.whl')
    parts = base.split('-')
    # format : name-version-python-abi-platform
    return {
        'name': parts[0],
        'version': parts[1],
        'python': parts[2],
        'abi': parts[3],
        'platform': parts[4],
    }

print(info_wheel('mon_outil-0.1.0-py3-none-any.whl'))
```

</details>

### Exercice 3 — SemVer *(facile)*

Écrire `prochaine_minor(version: str) -> str` qui prend `'0.1.0'` et renvoie `'0.2.0'`. (On reset le patch à 0 après une bump minor.)

In [ ]:
# Votre code ici


In [ ]:
# ▶ Une fois votre solution écrite ci-dessus, exécutez cette cellule
# pour signaler à votre formateur que vous avez tenté l'exercice.
import sys
from pathlib import Path
for _p in (Path.cwd(), *Path.cwd().parents):
    if (_p / "_common" / "utils_pedagogie.py").exists():
        sys.path.insert(0, str(_p / "_common")); break
from utils_pedagogie import marquer_tentative
marquer_tentative(notebook="07_Build_wheel", exercice=3)


<details>
<summary>📖 Voir la correction</summary>

```python
def prochaine_minor(version: str) -> str:
    """Incrémente la partie minor, reset patch."""
    major, minor, _patch = version.split('.')
    return f'{major}.{int(minor) + 1}.0'

print(prochaine_minor('0.1.0'))
print(prochaine_minor('1.4.7'))
```

</details>

### Exercice 4 — Version dynamique *(moyen)*

Écrire `extraire_version_pyproject(contenu: str) -> str` qui lit un contenu de `pyproject.toml` (fourni comme chaîne) et renvoie la valeur de `version = ...`. Parser à la main (sans `tomllib`).

In [ ]:
# Votre code ici


In [ ]:
# ▶ Une fois votre solution écrite ci-dessus, exécutez cette cellule
# pour signaler à votre formateur que vous avez tenté l'exercice.
import sys
from pathlib import Path
for _p in (Path.cwd(), *Path.cwd().parents):
    if (_p / "_common" / "utils_pedagogie.py").exists():
        sys.path.insert(0, str(_p / "_common")); break
from utils_pedagogie import marquer_tentative
marquer_tentative(notebook="07_Build_wheel", exercice=4)


<details>
<summary>📖 Voir la correction</summary>

```python
def extraire_version_pyproject(contenu: str) -> str:
    """Extrait la ligne `version = "..."` du pyproject.toml."""
    for ligne in contenu.splitlines():
        ligne = ligne.strip()
        if ligne.startswith('version'):
            _, _, apres = ligne.partition('=')
            return apres.strip().strip('"')
    return ''

exemple = '''
[project]
name = "mon-outil"
version = "0.1.0"
'''
print(extraire_version_pyproject(exemple))
```

</details>

### Exercice 5 — Version avec `tomllib` *(moyen)*

Même exercice, mais en utilisant `tomllib` (stdlib Python 3.11+). Écrire `extraire_version(chemin: str) -> str`.

In [ ]:
# Votre code ici


In [ ]:
# ▶ Une fois votre solution écrite ci-dessus, exécutez cette cellule
# pour signaler à votre formateur que vous avez tenté l'exercice.
import sys
from pathlib import Path
for _p in (Path.cwd(), *Path.cwd().parents):
    if (_p / "_common" / "utils_pedagogie.py").exists():
        sys.path.insert(0, str(_p / "_common")); break
from utils_pedagogie import marquer_tentative
marquer_tentative(notebook="07_Build_wheel", exercice=5)


<details>
<summary>📖 Voir la correction</summary>

```python
import tomllib
from pathlib import Path

def extraire_version(chemin: str) -> str:
    """Extrait project.version avec tomllib."""
    with open(chemin, 'rb') as f:
        data = tomllib.load(f)
    return data['project']['version']

Path('/tmp/pp.toml').write_text(
    '[project]\nname = "x"\nversion = "0.2.3"\n',
    encoding='utf-8',
)
print(extraire_version('/tmp/pp.toml'))
```

</details>

### Exercice 6 — Pipeline complet *(difficile)*

Écrire (en commentaires bash) les étapes d'un pipeline complet : 
1. lint avec ruff ;
2. tests avec pytest ;
3. build du wheel avec uv ;
4. upload sur TestPyPI.

In [ ]:
# Votre code ici


In [ ]:
# ▶ Une fois votre solution écrite ci-dessus, exécutez cette cellule
# pour signaler à votre formateur que vous avez tenté l'exercice.
import sys
from pathlib import Path
for _p in (Path.cwd(), *Path.cwd().parents):
    if (_p / "_common" / "utils_pedagogie.py").exists():
        sys.path.insert(0, str(_p / "_common")); break
from utils_pedagogie import marquer_tentative
marquer_tentative(notebook="07_Build_wheel", exercice=6)


<details>
<summary>📖 Voir la correction</summary>

```python
pipeline = '''
# 1. Lint
uv run ruff check .
uv run ruff format --check .

# 2. Tests
uv run pytest -v

# 3. Build
uv build
ls dist/

# 4. Upload TestPyPI
uv publish --publish-url https://test.pypi.org/legacy/

# 5. Vérification : installation depuis TestPyPI dans un venv neuf
uv pip install --index-url https://test.pypi.org/simple/ mon-outil
'''
print(pipeline)
```

</details>

---


## Ressources externes

- [`uv build`](https://docs.astral.sh/uv/concepts/projects/build/)
- [Wheel — PEP 427](https://peps.python.org/pep-0427/)
- [Packaging User Guide — tutoriel](https://packaging.python.org/en/latest/tutorials/packaging-projects/)
- [TestPyPI](https://test.pypi.org/)
- [SemVer](https://semver.org/lang/fr/)

---

## Mini-exemples supplémentaires

### Checklist avant release

- [ ] Version bumpée dans `pyproject.toml`
- [ ] `CHANGELOG.md` à jour
- [ ] `README.md` à jour
- [ ] Tag Git créé (`git tag v0.1.0 && git push --tags`)
- [ ] Lint vert (`ruff check .`)
- [ ] Tests verts (`pytest`)
- [ ] `uv build` réussi
- [ ] Test d'installation depuis le wheel
- [ ] `uv publish` (TestPyPI d'abord !)

### Inspecter un wheel

In [ ]:
# Un wheel est un zip — on peut le décompresser
# unzip -l dist/mon_outil-0.1.0-py3-none-any.whl
import zipfile
import io

# Créer un zip fictif pour illustrer
buf = io.BytesIO()
with zipfile.ZipFile(buf, 'w') as z:
    z.writestr('mon_outil/__init__.py', '# package')
    z.writestr('mon_outil-0.1.0.dist-info/METADATA', 'Name: mon-outil\nVersion: 0.1.0')

buf.seek(0)
with zipfile.ZipFile(buf) as z:
    for name in z.namelist():
        print(name)

### `.dist-info/METADATA`

Fichier texte à l'intérieur du wheel qui contient les métadonnées (PEP 427/PEP 621). Lisible directement :

In [ ]:
metadata = '''
Metadata-Version: 2.1
Name: mon-outil
Version: 0.1.0
Summary: Un petit utilitaire de démonstration.
Requires-Python: >=3.13
Requires-Dist: httpx>=0.27
Requires-Dist: rich>=13
'''
print(metadata)

### `__version__` synchronisé avec `pyproject.toml`

In [ ]:
# src/mon_outil/__init__.py
# from importlib.metadata import version
# __version__ = version('mon-outil')

from importlib.metadata import version
print(version('pip'))  # exemple : une lib déjà installée

Ce pattern évite de dupliquer le numéro de version dans le code. Il est lu depuis les métadonnées du wheel installé.

### Publier depuis GitHub Actions (aperçu)

```yaml
- name: Build and publish
  run: |
    uv build
    uv publish
  env:
    UV_PUBLISH_TOKEN: ${{ secrets.PYPI_API_TOKEN }}
```

Déclenché sur un tag Git : `on: push: tags: [ 'v*' ]`.

### Ne pas commiter `dist/`

In [ ]:
gitignore = '''
dist/
build/
*.egg-info/
.venv/
__pycache__/
*.pyc
'''
print(gitignore)

Ces dossiers sont des **artefacts**. Ils doivent être générés, pas versionnés.

---

## Quiz flash — vérifiez vos acquis

Ce quiz est là pour que vous vérifiiez rapidement votre compréhension avant de passer au notebook suivant. Les réponses sont dans le bloc `<details>` en dessous.


**Question 1.** Commande pour construire un wheel avec `uv` ?

<details>
<summary>📖 Réponse</summary>

`uv build`.

</details>

**Question 2.** Différence wheel ↔ sdist ?

<details>
<summary>📖 Réponse</summary>

Wheel (`.whl`) : package prêt à installer. Sdist (`.tar.gz`) : sources à rebuilder à l'installation.

</details>

**Question 3.** Que signifie `py3-none-any` dans le nom du wheel ?

<details>
<summary>📖 Réponse</summary>

Python 3 pur, sans code binaire, indépendant de la plateforme.

</details>

**Question 4.** Pourquoi tester sur TestPyPI avant PyPI ?

<details>
<summary>📖 Réponse</summary>

TestPyPI est un bac à sable identique à PyPI. Une publication ratée ne pollue pas le registre public.

</details>

**Question 5.** Quelle convention pour les numéros de version ?

<details>
<summary>📖 Réponse</summary>

SemVer : `MAJOR.MINOR.PATCH`.

</details>

---

## Cheat sheet — Release en 10 étapes

```bash
# 1. Mettre à jour la version
vim pyproject.toml   # version = "0.2.0"

# 2. Changelog
vim CHANGELOG.md

# 3. Commit
git add pyproject.toml CHANGELOG.md
git commit -m 'release 0.2.0'

# 4. Tag
git tag v0.2.0

# 5. Push
git push && git push --tags

# 6. Lint + tests
uv run ruff check .
uv run pytest

# 7. Build
rm -rf dist/
uv build
ls dist/

# 8. Install local (dans un venv propre)
uv pip install dist/*.whl

# 9. TestPyPI
uv publish --publish-url https://test.pypi.org/legacy/

# 10. PyPI
uv publish
```

### Anatomie d'un wheel

```
mon_outil-0.2.0-py3-none-any.whl   (c'est un zip)
├── mon_outil/
│   ├── __init__.py
│   ├── cli.py
│   └── core.py
└── mon_outil-0.2.0.dist-info/
    ├── METADATA
    ├── WHEEL
    ├── entry_points.txt
    └── RECORD
```

### Nom de fichier — comprendre chaque partie

`mon_outil-0.2.0-py3-none-any.whl`

| Partie | Sens |
|---|---|
| `mon_outil` | Nom du package (normalisé) |
| `0.2.0` | Version |
| `py3` | Python 3 (tout `py3.x` compatible) |
| `none` | Pas d'extension binaire |
| `any` | Toutes plateformes |

In [ ]:
# Exemple de `__version__` lu depuis les métadonnées
# (à mettre dans src/mon_outil/__init__.py)
from importlib.metadata import version, PackageNotFoundError

try:
    __version__ = version('mon-outil')
except PackageNotFoundError:
    __version__ = '0.0.0-dev'

print('__version__ défini avec fallback dev')

### Et si `uv` n'est pas disponible ?

Alternative officielle : le package `build` :

```bash
pip install build
python -m build
```

Pour publier, `twine` :

```bash
pip install twine
twine upload dist/*
```

Résultat identique à `uv build` + `uv publish`, mais plus lent.

### Hatchling : fichiers inclus / exclus

In [ ]:
config = '''
[tool.hatch.build.targets.wheel]
packages = ["src/mon_outil"]

[tool.hatch.build.targets.sdist]
exclude = [
    "/.github",
    "/tests",
    "/docs",
]
'''
print(config)

### Chaîne de commandes recommandée pour CI

```yaml
# .github/workflows/release.yml (extrait)
on:
  push:
    tags: [ 'v*' ]

jobs:
  release:
    runs-on: ubuntu-latest
    steps:
      - uses: actions/checkout@v4
      - uses: astral-sh/setup-uv@v3
      - run: uv build
      - run: uv publish
        env:
          UV_PUBLISH_TOKEN: ${{ secrets.PYPI_API_TOKEN }}
```